In [2]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#import math
from datetime import datetime, timedelta
#from scipy.stats import loguniform, randint, uniform, skew
#import statsmodels.api as sm
#from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_s
#from sklearn.model_selection import cross_val_score, StratifiedKFold, Randomize
#from sklearn.preprocessing import StandardScaler
#from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingCl

#import optuna
#import shap

# Set Display Preferences
pd.set_option('display.max_columns', None)
pd.options.display.float_format = "{:.2f}".format
np.random.seed(1)

In [3]:
### UK Based ecom retailer sales from Jan 12, 2009 to Sep 12, 2011.
### Mainly sells unique all-occasion gift-ware.
aisles = pd.read_csv("data/raw/aisles.csv")
departments = pd.read_csv("data/raw/departments.csv")
order_products__prior = pd.read_csv("data/raw/order_products__prior.csv")
    # contains the products for each past order & then the test & train datasets are for the customers final order
    # len = 32.4M rows (train len only 1.4M & doesnt include eval_set = test)
    # Neither order_products_x contain eval_set = test
    # Does reordered mean that customer has purchased that product in the past. if first purchase, reorder still = 0 instead of null
    # The goal of the original sompetition was to predict which previously purchased products will be in a user’s next order.
order_products__train = pd.read_csv("data/raw/order_products__train.csv")
    # len = 1.38M rows
    # The goal of the original competition was to predict which previously purchased products will be in a user’s next order.
        # so customer i has purchased x different products. objectice is to calc/predict which of those x items shows up in their final order.
        # achieved by calculating a proability (forecast?) for each previous item & if prob > a threshold then predict it will be in final order
            # note: so do we not care about the new items in the final order?????
orders = pd.read_csv("data/raw/orders.csv")
    # contains all orders by all customers in but the prior, train & test sets
    # a customers last order is categorized into train or test
    # a customers first order (order_num=1) has a blank days_since_prior_order which makes sense. sometimes users make multiple orders in 1 day making value of this factor = 0.
        # note: there are a lot of same day orders -> probably dont repeate same products in 1 day
    # max days since reorder is 30 days so 30 actually = 30+
products = pd.read_csv("data/raw/products.csv")


In [13]:
departments

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol
5,6,international
6,7,beverages
7,8,pets
8,9,dry goods pasta
9,10,bulk


In [ ]:
order_products = pd.concat([order_products__prior, order_products__train], ignore_index=True)
df = orders.merge(order_products, how='inner', on='order_id')
df = df.merge(products, how='left', on='product_id')
df = df.merge(aisles, how='left', on='aisle_id')
df = df.merge(departments, how='left', on='department_id')

# One-hot encode day of week column
df['order_day'] = df['order_dow']+1
df = pd.get_dummies(df, columns=['order_day'], dtype=int)
# One-hot encode departments
df = pd.get_dummies(df, columns=['department'], dtype=int)

# Capture names of all dummy columns (useful later for groupby)
dummy_cols = [c for c in df.columns if c.startswith('order_day_') or c.startswith('department_')]

df

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,order_day_1,order_day_2,order_day_3,order_day_4,order_day_5,order_day_6,order_day_7,department_alcohol,department_babies,department_bakery,department_beverages,department_breakfast,department_bulk,department_canned goods,department_dairy eggs,department_deli,department_dry goods pasta,department_frozen,department_household,department_international,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,soft drinks,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,popcorn jerky,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,popcorn jerky,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33819101,272231,206209,train,14,6,14,30.00,40603,4,0,Fabric Softener Sheets,75,17,laundry,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
33819102,272231,206209,train,14,6,14,30.00,15655,5,0,Dark Chocolate Mint Snacking Chocolate,45,19,candy chocolate,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
33819103,272231,206209,train,14,6,14,30.00,42606,6,0,Phish Food Frozen Yogurt,37,1,ice cream ice,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
33819104,272231,206209,train,14,6,14,30.00,37966,7,0,French Baguette Bread,112,3,bread,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [42]:
dummy_cols

['department_id',
 'order_day_1',
 'order_day_2',
 'order_day_3',
 'order_day_4',
 'order_day_5',
 'order_day_6',
 'order_day_7',
 'department_alcohol',
 'department_babies',
 'department_bakery',
 'department_beverages',
 'department_breakfast',
 'department_bulk',
 'department_canned goods',
 'department_dairy eggs',
 'department_deli',
 'department_dry goods pasta',
 'department_frozen',
 'department_household',
 'department_international',
 'department_meat seafood',
 'department_missing',
 'department_other',
 'department_pantry',
 'department_personal care',
 'department_pets',
 'department_produce',
 'department_snacks']

In [39]:
# Calc avg basket size & std dev of avg basket size
basket_sizes = df.groupby(['user_id', 'order_number']).size().reset_index(name='basket_size')
basket_var = basket_sizes.groupby('user_id').agg(
                                            avg_basket_size=('basket_size', 'mean'),
                                            std_basket_size=('basket_size', 'std'))

In [43]:
df_user_id = df.groupby(['user_id'], as_index=False).agg(
            # order_id
            # eval_set
            # order_number
                orders = ('order_number', 'nunique'),
                hundred_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_number'] >= 100].nunique()),               # flag if customer has 100+ orders
            # order_dow
                unqiue_days = ('order_dow', 'nunique'),                                                                         # number of distinct days of week a customer ordered on
                weekend_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_dow'] <= 1].nunique()),                    # weekend ordered
                weekday_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_dow'] > 1].nunique()),                     # week ordered
            # order_hour_of_day
                evening_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_hour_of_day'] >= 18].nunique()),
                afternoon_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_hour_of_day'] >= 12].nunique()),
                morning_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_hour_of_day'] < 12].nunique()),
            # product_id
                unqiue_products = ('product_id', 'nunique'),
            # add_to_cart_order
            # reordered
            # aisle_id
                unqiue_aisles = ('aisle_id', 'nunique'),
            # department_id
                unqiue_departments = ('department_id', 'nunique'),

            **{col: (col, 'mean') for col in dummy_cols}        # Dynamically unpacks as (column, function) tuples instead of a dictionary mapping
            )

df_user_id



,user_id,orders,hundred_orders,unqiue_days,weekend_orders,weekday_orders,evening_orders,afternoon_orders,morning_orders,unqiue_products,unqiue_aisles,unqiue_departments,department_id,order_day_1,order_day_2,order_day_3,order_day_4,order_day_5,order_day_6,order_day_7,department_alcohol,department_babies,department_bakery,department_beverages,department_breakfast,department_bulk,department_canned goods,department_dairy eggs,department_deli,department_dry goods pasta,department_frozen,department_household,department_international,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks
0,1,11,0,4,3,8,0,4,7,19,13,7,14.17,0.00,0.24,0.13,0.16,0.47,0.00,0.00,0.00,0.00,0.00,0.21,0.06,0.00,0.00,0.24,0.00,0.00,0.00,0.04,0.00,0.00,0.00,0.00,0.01,0.00,0.00,0.07,0.36
1,2,15,0,5,6,9,0,2,13,121,37,13,12.05,0.00,0.42,0.38,0.13,0.04,0.03,0.00,0.00,0.00,0.01,0.04,0.01,0.00,0.02,0.22,0.11,0.00,0.12,0.00,0.01,0.00,0.00,0.00,0.05,0.01,0.00,0.19,0.21
2,3,12,0,4,8,4,3,12,0,33,16,9,9.44,0.52,0.18,0.06,0.24,0.00,0.00,0.00,0.00,0.00,0.00,0.03,0.00,0.00,0.00,0.24,0.02,0.05,0.07,0.01,0.00,0.00,0.00,0.00,0.05,0.00,0.00,0.43,0.10
3,4,5,0,3,0,5,0,3,2,17,14,9,8.67,0.00,0.00,0.00,0.00,0.50,0.28,0.22,0.11,0.00,0.11,0.17,0.00,0.00,0.06,0.00,0.11,0.00,0.17,0.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.11,0.06
4,5,5,0,3,3,2,2,4,1,28,17,10,8.28,0.39,0.26,0.00,0.35,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.20,0.02,0.02,0.04,0.02,0.09,0.00,0.00,0.00,0.07,0.00,0.00,0.50,0.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206204,206205,4,0,4,1,3,0,4,0,37,20,11,11.43,0.00,0.37,0.16,0.00,0.33,0.14,0.00,0.00,0.02,0.04,0.02,0.00,0.00,0.00,0.39,0.10,0.00,0.06,0.00,0.02,0.02,0.04,0.00,0.02,0.00,0.00,0.27,0.00
206205,206206,67,0,7,26,41,31,66,1,150,50,16,9.39,0.24,0.19,0.12,0.16,0.17,0.04,0.09,0.00,0.00,0.01,0.12,0.00,0.00,0.03,0.14,0.02,0.00,0.27,0.04,0.01,0.01,0.00,0.00,0.05,0.04,0.00,0.12,0.15
206206,206207,16,0,7,6,10,3,9,7,92,46,14,10.69,0.13,0.19,0.16,0.18,0.03,0.13,0.17,0.00,0.00,0.01,0.09,0.02,0.00,0.05,0.23,0.04,0.04,0.09,0.00,0.00,0.03,0.00,0.00,0.07,0.00,0.00,0.22,0.10
206207,206208,49,0,7,15,34,9,37,12,198,63,17,10.31,0.08,0.18,0.26,0.15,0.13,0.12,0.09,0.00,0.00,0.09,0.03,0.03,0.00,0.02,0.24,0.04,0.03,0.04,0.01,0.00,0.03,0.00,0.00,0.06,0.01,0.00,0.29,0.09


In [ ]:
# Returns Percent
df_userid['return_pct'] = -1*df_userid['return_amt'] / (-1*df_userid['return_amt'] + df_userid['sale_amt'])
# Units per Transaction
df_aggregated['upt'] = df_aggregated['sale_qty'] / df_aggregated['sale_txns']
# Average Unit Retail
df_aggregated['aur'] = df_aggregated['sale_amt'] / df_aggregated['sale_qty']          
# Average Order Value
aggregated['aov'] = df_aggregated['sale_amt'] / df_aggregated['sale_txns']
# SKU Mix per Order
df_aggregated['sku_mix'] = df_aggregated['total_skus'] / df_aggregated['total_txns']

df

In [ ]:
df_minus_firstorder = df[df['order_number']!=0]
reorder_rate = df_minus_firstorder.groupby(['user_id'], as_index=False).agg(
                                                                        reorder_rate = ('reordered', 'mean'),
                                                                        reorder_frequency_std = ('days_since_prior_order', 'std'),

                                                                        avg_reorder_size=('order_id', lambda x: df.loc[x.index, 'reordered'].sum() / x.nunique()),

                                                                        same_day_repurchase = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] == 0].nunique()),
                                                                        same_week_repurchase = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] > 7].nunique()),

                                                                        days_since_over30 = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] >= 30].nunique()),
                                                                        days_since_under30 = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] < 30].nunique())

                                                                        )
reorder_rate

,user_id,reorder_rate,reorder_frequency_std,avg_reorder_size,days_since_over30,days_since_under30
0,1,0.73,8.79,4.64,1,9
1,2,0.46,9.78,7.00,3,11
2,3,0.62,4.87,4.58,0,11
3,4,0.06,8.58,0.20,0,4
4,5,0.39,5.25,3.60,0,4
...,...,...,...,...,...,...
206204,206205,0.27,8.61,3.50,1,2
206205,206206,0.47,3.45,2.01,0,66
206206,206207,0.59,11.29,8.19,4,11
206207,206208,0.71,4.02,9.78,0,48


In [43]:

df.isna().sum()

order_id                        0
user_id                         0
eval_set                        0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
product_id                      0
add_to_cart_order               0
reordered                       0
product_name                    0
aisle_id                        0
department_id                   0
aisle                           0
department                      0
dtype: int64

In [ ]:
# so if we are clustering (based on previous purchase history??) then what are some factors we might want to consider?
    # number of orders
    # avg time since last order & std dev of time between orders
    # avg number of items per order & std dev



    # categories:
        # weekly purchaser (all purchases 7 days apart)? or always on same day of week (or maybe weekened vs weekday shopers or maybe just percentage split)
        # single product/category customers?


<StringArray>
['prior', 'train', 'test']
Length: 3, dtype: str

In [31]:
df['product_name'].unique()

<StringArray>
[                                                     'Soda',
                   'Organic Unsweetened Vanilla Almond Milk',
                                       'Original Beef Jerky',
                                'Aged White Cheddar Popcorn',
                          'XL Pick-A-Size Paper Towel Rolls',
                                                'Pistachios',
                                    'Bag of Organic Bananas',
                                     'Cinnamon Toast Crunch',
                                     'Organic String Cheese',
                                      'Creamy Almond Butter',
 ...
                                        'Peachtree Schnapps',
                                    'Hennepin Farmhouse Ale',
                                         'Cld/Flu Van Chrry',
 'Lowfat Cherry Lime Supernova Kefir Cultured Milk Smoothie',
                                     '18 Year Scotch Whisky',
  'Training  Fluoride Free Toothpaste Apple & Banan

In [ ]:
# INPUTS: cluster on composition, style, and timing features that are roughly invariant to how much someone uses the product
    #
    #
    #
    #
    #



# OUTPUTS: variable used as a downstream outcome is excluded from the inputs; 
    # volume: number of items
    # frequency & legacy???
    #
# Note: related-but-distinct variables may remain but are expected to show partial mechanical correlation, which is disclosed.